[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/06_residual_and_connections.ipynb)

# 06. Residual and connection structures

단순 residual에서 gate와 다중 stream mixing으로 구조가 복잡해지는 과정을 작은 tensor로 본다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Plain residual

기본 형태는 x + F(x).


In [ ]:
x = torch.tensor([[1., 2., 3., 4.]], device=device)
branch = nn.Linear(4, 4, bias=False).to(device)
with torch.no_grad():
    branch.weight.copy_(0.1 * torch.eye(4, device=device))

y = x + branch(x)
print(y)


In [ ]:
_ = profile_call("plain residual", lambda z: z + branch(z), x)


## 2. ReZero / LayerScale

branch를 scalar 또는 channel scale로 조절한다.


In [ ]:
alpha = torch.tensor(0.1, device=device)
gamma = torch.tensor([0.1, 0.2, 0.3, 0.4], device=device)

rezero = x + alpha * branch(x)
layerscale = x + gamma * branch(x)

print("ReZero:", rezero)
print("LayerScale:", layerscale)


In [ ]:
_ = profile_call("LayerScale", lambda z: z + gamma * branch(z), x)


## 3. Gated residual

sigmoid gate가 residual branch의 기여도를 제어한다.


In [ ]:
gate_logit = torch.tensor([[-2., -1., 1., 2.]], device=device)
gate = torch.sigmoid(gate_logit)
y = x + gate * branch(x)

print("gate:", gate)
print("output:", y)


In [ ]:
_ = profile_call("gated residual", lambda z: z + gate * branch(z), x)


## 4. Multi-stream connection

두 residual stream 사이를 작은 mixing matrix로 섞는다.


In [ ]:
streams = torch.tensor(
    [[[1., 0., 1., 0.],
      [0., 1., 0., 1.]]],
    device=device,
)
mix = torch.tensor([[0.8, 0.2], [0.3, 0.7]], device=device)

mixed = torch.einsum("ij,bjd->bid", mix, streams)
print("mixed streams:\n", mixed)


In [ ]:
_ = profile_call("stream mixing", lambda s, m: torch.einsum("ij,bjd->bid", m, s), streams, mix)


## 5. Doubly-stochastic mixing idea

mHC 계열의 핵심 제약을 작은 Sinkhorn normalization으로 확인한다.


In [ ]:
logits = torch.tensor([[2., 0.], [1., 3.]], device=device)
M = logits.exp()

for _ in range(4):
    M = M / M.sum(dim=1, keepdim=True)
    M = M / M.sum(dim=0, keepdim=True)

print("mixing matrix:\n", M)
print("row sums:", M.sum(1))
print("col sums:", M.sum(0))


In [ ]:
_ = profile_call("Sinkhorn mixing", lambda z: (z.exp() / z.exp().sum(1, keepdim=True)), logits)


## References and provenance

**[6.1] Residual connection**
- 출처: He et al., Deep Residual Learning
- 이 노트북에서 가져온 부분: identity shortcut

**[6.2] ReZero**
- 출처: Bachlechner et al., ReZero is All You Need
- 이 노트북에서 가져온 부분: learned residual scalar

**[6.3] LayerScale**
- 출처: Touvron et al., Going Deeper With Image Transformers
- 이 노트북에서 가져온 부분: channel-wise residual scale

**[6.4] Hyper-Connections / mHC**
- 출처: DeepSeek research on Hyper-Connections and mHC
- 이 노트북에서 가져온 부분: multi-stream residual mixing and constrained mixing
